# 03 — Paired Randomization Test for System Outputs

Use a paired randomization test when two systems produce outputs for the same examples and the metric is computed over the whole corpus.

This is common in machine translation and other generation settings. The notebook uses a small dummy MT-style dataset and a simple corpus BLEU implementation for teaching purposes.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make src/ importable when running from the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from significance_utils import *

DATA_DIR = PROJECT_ROOT / 'data' / 'dummy'
ALPHA = 0.05

## Student input

In [ ]:
ALPHA = 0.05
N_PERMUTATIONS = 5000
DATA_FILE = DATA_DIR / 'mt_outputs.csv'

In [ ]:
df = pd.read_csv(DATA_FILE)
df.head()

## Compute the observed corpus scores

In [ ]:
bleu_a = simple_corpus_bleu(df['model_a_output'].tolist(), df['reference'].tolist())
bleu_b = simple_corpus_bleu(df['model_b_output'].tolist(), df['reference'].tolist())

print(f'Model A simple BLEU: {bleu_a:.2f}')
print(f'Model B simple BLEU: {bleu_b:.2f}')
print(f'Difference B - A: {bleu_b - bleu_a:.2f}')

## Run the paired randomization test

Under the null hypothesis, the labels "Model A" and "Model B" are exchangeable for each example. The test repeatedly swaps outputs for random subsets of examples and recomputes the corpus-level score difference.

In [ ]:
result = paired_randomization_test(
    outputs_a=df['model_a_output'],
    outputs_b=df['model_b_output'],
    references=df['reference'],
    metric_fn=simple_corpus_bleu,
    n_permutations=N_PERMUTATIONS,
    seed=7,
)

pd.Series(result)

In [ ]:
print(interpret_p_value(result['p_value'], alpha=ALPHA))

## Inspect some paired outputs

In [ ]:
for _, row in df.sample(5, random_state=1).iterrows():
    print('Reference:', row['reference'])
    print('Model A:  ', row['model_a_output'])
    print('Model B:  ', row['model_b_output'])
    print()

## Exercise questions

1. Why is randomization useful for corpus-level metrics?
2. What is being swapped in this test?
3. Why is the simple BLEU implementation here only for teaching, not publication-quality MT evaluation?